In [1]:
import torch
import torch.nn as nn  
import numpy as np  


In [ ]:
class Embeddings(nn.Module):  
  
    def __init__(self, vocab_size, dm, pad_id) -> None:  
        super().__init__()
        # 标量的token_ids --> embedding
        self.embedding = nn.Embedding(vocab_size, dm, padding_idx=pad_id)  
  
    def forward(self, x):  
        # inshape: x - (batch_size, seq_len,)  
        # embed tokens to dm | shape: out - (batch_size, seq_len, dm)  
        out = self.embedding(x)  
        return out

class PositionalEncoder(nn.Module):  
  
    def __init__(self, dm, maxlen, dropout=0.1, scale=True) -> None:  
        super().__init__()  
        self.dm = dm  
        self.drop = nn.Dropout(dropout)  
        self.scale = scale  
        # shape: pos - (maxlen, 1) dim - (dm, )  
        pos = torch.arange(maxlen).float().unsqueeze(1)
        dim = torch.arange(dm).float()
        # apply pos / (10000^2*i / dm) -> use sin for even indices & cosine for odd indices  
        values = pos / torch.pow(1e4, 2 * torch.div(dim, 2, rounding_mode="floor") / dm)  
        encodings = torch.where(dim.long() % 2 == 0, torch.sin(values), torch.cos(values))  
        # reshape: encodings - (1, maxlen, dm)  
        encodings = encodings.unsqueeze(0)  
          
        # register encodings w/o grad  
        self.register_buffer("pos_encodings", encodings)  
  
    def forward(self, embeddings):  
        # inshape: embeddings - (batch_size, seq_len, dm)  
        # scale embeddings (if applicable)  
        if self.scale:  
            embeddings = embeddings * np.sqrt(self.dm)  
        # sum embeddings w/ respective positonal encodings | shape: embeddings - (batch_size, seq_len, dm)  
        seq_len = embeddings.size(1)  
        embeddings = embeddings + self.pos_encodings[:, :seq_len]  
        # drop neurons | out - (batch_size, seq_len, dm)  
        out = self.drop(embeddings)  
        return out


In [ ]:
import torch  
import torch.nn as nn  
import numpy as np  
  
class ScaledDotProductAttention(nn.Module):  
  
    def __init__(self, dk, dropout=None) -> None:  
        super().__init__()  
        self.dk = dk  
        self.drop = nn.Dropout(dropout)  
        self.softmax = nn.Softmax(dim=-1)  
          
    def forward(self, q, k, v, mask=None):  
        # inputs are projected | shape: q - (batch_size, *n_head, q_len, dk) k - (batch_size, *n_head, k_len, dk)  v - (batch_size, *n_head, k_len, dv)  
  
        # compute dot prod w/ q & k then scale | shape: similarities - (batch_size, *n_head, q_len, k_len)  
        similarities = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.dk)  
  
        # apply mask (if required)  
        if mask is not None:  
            mask = mask.unsqueeze(1) # for multi-head attention  
            similarities = similarities.masked_fill(mask == 0,-1e9)  
  
        # compute attention weights | shape: attention - (batch_size, *n_head, q_len, k_len)  
        attention = self.softmax(similarities)  
        # drop attention weights  
        attention = self.drop(attention)  
  
        # compute context given v | shape: context - (batch_size, *n_head, q_len, dv)  
        context = torch.matmul(attention, v)  
        return context, attention


In [ ]:
class MultiHeadAttention(nn.Module):  
  
    def __init__(self, dm, dk, dv, nhead, bias=False, dropout=None) -> None:  
        super().__init__()  
        if dm % nhead != 0:  
            raise ValueError("Embedding dimensions (dm) must be evenly divisble by number of heads (nhead)")  
        self.dm = dm  
        self.dk = dk  
        self.dv = dv  
        self.nhead = nhead  
        self.wq = nn.Linear(dm, dk * nhead, bias=bias)  
        self.wk = nn.Linear(dm, dk * nhead, bias=bias)  
        self.wv = nn.Linear(dm, dv * nhead, bias=bias)  
        self.wo = nn.Linear(dv * nhead, dm)  
        self.scaled_dot_prod_attn = ScaledDotProductAttention(dk, dropout=dropout)  
  
    def forward(self, q, k, v, mask=None):  
        # inshape: q - (batch_size, q_len, dm) k & v - (batch_size, k_len, dm)  
        batch_size, q_len, k_len = q.size(0), q.size(1), k.size(1)  
  
        # linear projections into heads | shape: q - (batch_size, nhead, q_len, dk) k - (batch_size, nhead, k_len, dk) v - (batch_size, nhead, k_len, dv)  
        q = self.wq(q).view(batch_size, q_len, self.nhead, self.dk).transpose(1, 2)  
        k = self.wk(k).view(batch_size, k_len, self.nhead, self.dk).transpose(1, 2)  
        v = self.wv(v).view(batch_size, k_len, self.nhead, self.dv).transpose(1, 2)  
  
        # get context & attn weights | shape: attention - (batch_size, nhead, q_len, k_len) context - (batch_size, nhead, q_len, dv)  
        context, attention = self.scaled_dot_prod_attn(q, k, v, mask=mask)  
  
        # concat heads | shape: context - (batch_size, q_len, dm)  
        context = context.transpose(1, 2).contiguous().view(batch_size, q_len, self.dm)  
  
        # project context vector | shape: context - (batch_size, q_len, dm)  
        context = self.wo(context)  
        return context, attention


In [5]:
def generate_pad_mask(seq, pad_id):  
    # inshape: seq - (batch_size, seq_len)  
  
    # mark non-pad True & pad False   
    mask = (seq != pad_id).unsqueeze(-2)  
    # outshape: mask - (batch_size, 1, seq_len)  
    return mask
  
def generate_nopeak_pad_mask(trg, pad_id):  
    # inshape: trg - (batch_size, trg_len)  
  
    # create pad mask (True = no pad False = pad) | shape: trg_mask - (batch_size, 1, trg_len)  
    trg_mask = generate_pad_mask(trg, pad_id)  
    # create subsequent mask | shape: trg_nopeak_mask - (1, trg_len, trg_len)  
    trg_len = trg.size(1)  
    trg_nopeak_mask = torch.triu(torch.ones((1, trg_len, trg_len)) == 1)  
    trg_nopeak_mask = trg_nopeak_mask.transpose(1, 2)  
    # combine pad & subsequent mask shape  
    trg_mask = trg_mask & trg_nopeak_mask  
    # outshape: trg_mask - (batch_size, trg_len, trg_len)  
    return trg_mask

def generate_masks(src, trg, pad_id):  
    # inshape: src - (batch_size, src_len) trg - (batch_size, trg_len)  
  
    # create pad mask for src (True = no pad False = pad)  
    src_mask = generate_pad_mask(src, pad_id)  
    # generate pad nopeak mask for trg  
    trg_mask = generate_nopeak_pad_mask(trg, pad_id)  
    # outshape: src_mask - (batch_size, 1, src_len) trg_mask - (batch_size, trg_len, trg_len)  
    return src_mask, trg_mask



In [11]:
a = torch.tensor([[1,2,3],[4,5,0]])
print(generate_masks(a, a, 0))

(tensor([[[ True,  True,  True]],

        [[ True,  True, False]]]), tensor([[[ True, False, False],
         [ True,  True, False],
         [ True,  True,  True]],

        [[ True, False, False],
         [ True,  True, False],
         [ True,  True, False]]]))


In [6]:
class FeedForwardNetwork(nn.Module):  
  
    def __init__(self, dm, dff, dropout=0.1) -> None:  
        super().__init__()  
        self.w1 = nn.Linear(dm, dff)  
        self.w2 = nn.Linear(dff, dm)  
        self.relu = nn.ReLU(inplace=False)  
        self.drop = nn.Dropout(dropout)  
  
    def forward(self, x):  
        # inshape: x - (batch_size, seq_len, dm)  
          
        # first linear transform with ReLU | shape: x - (batch_size, seq_len, dff)  
        x = self.relu(self.w1(x))  
        # drop neurons  
        x = self.drop(x)  
        # second linear transform | shape: out - (batch_size, seq_len, dm)  
        out = self.w2(x)  
        return out

class Norm(nn.Module):  
    """
    Layer Norm
    """
    def __init__(self, dm, eps=1e-6):  
        super().__init__()  
        self.gamma = nn.Parameter(torch.ones(dm))  
        self.beta = nn.Parameter(torch.zeros(dm))  
        self.eps = eps  
  
    def forward(self, x: torch.Tensor):  
        # inshape: x - (batch_size, seq_len, dm)  
  
        # calc mean & variance (along dm)  
        mean = x.mean(dim=-1, keepdim=True)  
        var = x.var(dim=-1, unbiased=True, keepdim=True)  
        # normalize, scale & shift | shape: out - (batch_size, seq_len, dm)  
        norm = (x - mean) / torch.sqrt(var + self.eps)  
        out = norm * self.gamma + self.beta  
        return out


In [ ]:
# from embedding import Embeddings  
# from pos_encoder import PositionalEncoder  
# from attention import MultiHeadAttention  
# from norm import Norm  
# from feedforward import FeedForwardNetwork  

class EncoderLayer(nn.Module):  
  
    def __init__(self, dm, dk, dv, nhead, dff, bias=False, dropout=0.1, eps=1e-6) -> None:  
        super().__init__()  
        self.multihead = MultiHeadAttention(dm, dk, dv, nhead, bias=bias, dropout=dropout)  
        self.feedforward = FeedForwardNetwork(dm, dff, dropout=dropout)  
        self.norm1 = Norm(dm, eps=eps)  
        self.norm2 = Norm(dm, eps=eps)  
        self.drop1 = nn.Dropout(dropout)  
        self.drop2 = nn.Dropout(dropout)  

    def forward(self, src, src_mask=None):  
        # inshape: src - (batch_size, src_len, dm)  
  
        # get context | shape - x_out (batch_size, src_len, dm)  
        x = src  
        x_out, attn = self.multihead(x, x, x, mask=src_mask)  
        # drop neurons  
        x_out = self.drop1(x_out)  
        # add & norm (residual connections) | shape: x - (batch_size, src_len, dm)  
        x = self.norm1(x + x_out)  
  
        # linear transforms | shape: x_out (batch_size, src_len, dm)  
        x_out = self.feedforward(x)   
        # drop neurons  
        x_out = self.drop2(x_out)  
        # add & norm (residual connections) | shape: out - (batch_size, src_len, dm)  
        out = self.norm2(x + x_out)  
        return out, attn  
  
class Encoder(nn.Module):  
  
    def __init__(self, vocab_size, maxlen, pad_id, dm, dk, dv, nhead, dff, layers=6, bias=False,   
                 dropout=0.1, eps=1e-6, scale=True) -> None:  
        super().__init__()  
        self.embeddings = Embeddings(vocab_size, dm, pad_id)  
        self.pos_encodings = PositionalEncoder(dm, maxlen, dropout=dropout, scale=scale)  
        self.stack = nn.ModuleList([EncoderLayer(dm, dk, dv, nhead, dff, bias=bias, dropout=dropout, eps=eps)   
                                    for l in range(layers)])  
  
    def forward(self, src, src_mask=None):  
        # inshape: src - (batch_size, src_len, dm) src_mask - (batch_size, 1, src_len)  
  
        # embeddings + positional encodings | shape: x - (batch_size, src_len, dm)  
        x = self.embeddings(src)  
        x = self.pos_encodings(x)  
        # pass src through stack of encoders (out of layer is in for next)  
        for encoder in self.stack:  
            x, attn = encoder(x, src_mask=src_mask)  
        # shape: out - (batch_size, src_len, dm)  
        out = x  
        return out, attn

# Decoder

In [ ]:
import torch.nn as nn  
# from embedding import Embeddings  
# from pos_encoder import PositionalEncoder  
# from attention import MultiHeadAttention  
# from norm import Norm  
# from feedforward import FeedForwardNetwork  
  
class DecoderLayer(nn.Module):  
  
    def __init__(self, dm, dk, dv, nhead, dff, bias=False, dropout=0.1, eps=1e-6) -> None:  
        super().__init__()  
        self.maskmultihead = MultiHeadAttention(dm, dk, dv, nhead, bias=bias, dropout=dropout)  
        self.multihead = MultiHeadAttention(dm, dk, dv, nhead, bias=bias, dropout=dropout)  
        self.feedforward = FeedForwardNetwork(dm, dff, dropout=dropout)  
        self.norm1 = Norm(dm, eps=eps)  
        self.norm2 = Norm(dm, eps=eps)  
        self.norm3 = Norm(dm, eps=eps)  
        self.drop1 = nn.Dropout(dropout)  
        self.drop2 = nn.Dropout(dropout)  
        self.drop3 = nn.Dropout(dropout)  
  
    def forward(self, src, trg, src_mask=None, trg_mask=None):  
        # inshape: src - (batch_size src_len, dm) trg - (batch_size, trg_len, dm) \  
        # src_mask - (batch_size, 1 src_len) trg_mask - (batch_size trg_len, trg_len)/(batch_size, 1 , trg_len)  
  
        # calc masked context | shape: x_out - (batch_size, trg_len, dm)  
        x = trg  
        x_out, attn1 = self.maskmultihead(x, x, x, mask=trg_mask)  
        # drop neurons  
        x_out = self.drop1(x_out)  
        # add & norm (residual connections) | shape: x - (batch_size, trg_len, dm)  
        x = self.norm1(x + x_out)  
  
        # calc context | shape: x_out - (batch_size, trg_len, dm)  
        x_out, attn2 = self.multihead(x, src, src, mask=src_mask)  
        # drop neurons  
        x_out = self.drop2(x_out)  
        # add & norm (residual connections) | shape: x - (batch_size, trg_len, dm)  
        x = self.norm2(x + x_out)  
  
        # calc linear transforms | shape: x_out - (batch_size, trg_len, dm)  
        x_out = self.feedforward(x)  
        # drop neurons  
        x_out = self.drop3(x_out)  
        # add & norm (residual connections) | shape: out - (batch_size, trg_len, dm)  
        out = self.norm3(x + x_out)  
        return out, attn1, attn2  
      
class Decoder(nn.Module):  
  
    def __init__(self, vocab_size, maxlen, pad_id, dm, dk, dv, nhead, dff, layers=6, bias=False,   
                 dropout=0.1, eps=1e-6, scale=True) -> None:  
        super().__init__()  
        self.embeddings = Embeddings(vocab_size, dm, pad_id)  
        self.pos_encodings = PositionalEncoder(dm, maxlen, dropout=dropout, scale=scale)  
        self.stack = nn.ModuleList([DecoderLayer(dm, dk, dv, nhead, dff, bias=bias, dropout=dropout, eps=eps)   
                                    for l in range(layers)])  
          
    def forward(self, src, trg, src_mask=None, trg_mask=None):  
        # inshape: src - (batch_size, src_len, dm) trg - (batch_size, trg_len, dm)  
  
        # embeddings + positional encodings | shape: x - (batch_size, trg_len, dm)  
        x = self.embeddings(trg)  
        x = self.pos_encodings(x)  
        # pass src & trg through stack of decoders (out of layer is in for next)  
        for decoder in self.stack:  
            x, attn1, attn2 = decoder(src, x, src_mask=src_mask, trg_mask=trg_mask)  
        out = x  
        return out, attn1, attn2


# Transformer

In [ ]:
import torch.nn as nn  
# from encoder import Encoder  
# from decoder import Decoder  
  
class Transformer(nn.Module):  
      
    def __init__(self, vocab_enc, vocab_dec, maxlen, pad_id, dm=512, dk=64, dv=64, nhead=8, layers=6,   
                dff=2048, bias=False, dropout=0.1, eps=1e-6, scale=True) -> None:  
        super().__init__()  
        self.encoder = Encoder(vocab_enc, maxlen, pad_id, dm, dk, dv, nhead, dff,   
                        layers=layers, bias=bias, dropout=dropout, eps=eps, scale=scale)            
        self.decoder = Decoder(vocab_dec, maxlen, pad_id, dm, dk, dv, nhead, dff,   
                        layers=layers, bias=bias, dropout=dropout, eps=eps, scale=scale)  
        self.linear = nn.Linear(dm, vocab_dec)  
        self.maxlen = maxlen  
        self.pad_id = pad_id  
        self.apply(xavier_init)  
  
    def forward(self, src, trg, src_mask=None, trg_mask=None):  
        # inshape: src - (batch_size, src_len) trg - (batch_size, trg_len)\  
        # src_mask - (batch_size, 1, src_len) trg_mask - (batch_size, 1, trg_len, trg_len)  
          
        # encode embeddings | shape: e_out - (batch_size, src_len, dm)  
        e_out, attn = self.encoder(src, src_mask=src_mask)  
  
        # decode embeddings | shape: d_out - (batch_size, trg_len, dm)  
        d_out, attn, attn = self.decoder(e_out, trg, src_mask=src_mask, trg_mask=trg_mask)  
        # linear transform decoder output | shape: out - (batch_size, trg_len, vocab_size)  
        out = self.linear(d_out)  
        return out  
  
def xavier_init(module):  
    if hasattr(module, "weight") and module.weight.dim() > 1:  
        init.xavier_uniform_(module.weight.data)


# Train

In [ ]:
import numpy as np  
import torch.nn as nn  
# from utils.functional import generate_masks  
  
def train(dataloader, model, optimizer, epochs=1000, device=None):  
    # setup  
    model.train()  
    m = len(dataloader)  
    cross_entropy = nn.CrossEntropyLoss(ignore_index=model.pad_id)  
    losses = []  
  
    # train over epochs  
    print("Training Started")  
    for epoch in range(epochs):  
        accum_loss = 0 # reset accumulative loss  
        for inputs, labels in dataloader:  
            # get src & trg  
            src, trg, out = inputs, labels[:, :-1], labels[:, 1:] # shape: src - (batch_size, src_len) trg & out - (batch_size, trg_len)  
            src, trg, out = src.long(), trg.long(), out.long()  
            # generate the masks  
            src_mask, trg_mask = generate_masks(src, trg, model.pad_id)  
            # move to device   
            src, trg, out = src.to(device), trg.to(device), out.to(device)  
            src_mask, trg_mask = src_mask.to(device), trg_mask.to(device)  
  
            # zero the grad  
            optimizer.zero_grad()  
            # get pred & reshape outputs  
            pred = model(src, trg, src_mask=src_mask, trg_mask=trg_mask) # shape: pred - (batch_size, seq_len, vocab_size)  
            pred, out = pred.contiguous().view(-1, pred.size(-1)), out.contiguous().view(-1) # shape: pred - (batch_size * seq_len, vocab_size) out - (batch_size * seq_len)  
            # calc grad & update model params  
            loss = cross_entropy(pred, out)  
            loss.backward()  
            optimizer.step()  
            # accumulate loss over time  
            accum_loss += loss.item()  
  
        # get epoch loss & keep track  
        epoch_loss = accum_loss / m  
        losses.append(epoch_loss)  
        print(f"Epoch {epoch + 1} Complete | Loss: {epoch_loss:.4f}")  
  
    # calc avg train loss  
    loss = np.mean(losses).item()  
    print(f"Training Complete | Average Loss: {loss:.4f}")  
    return loss
